In [35]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from model import GraphSAGE
import copy

checkpoint = torch.load('models/task_a_model.pt', weights_only=False)
data = checkpoint['data']
student_id_map = checkpoint['student_id_map']
course_id_map = checkpoint['course_id_map']
concept_id_map = checkpoint['concept_id_map']

model = GraphSAGE(hidden_dim=256)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

students = pd.read_csv('datasets/students.csv')
courses = pd.read_csv('datasets/courses.csv')
enrollments = pd.read_csv('datasets/enrollments.csv')
chatbot = pd.read_csv('datasets/chatbot_signals.csv')

course_name_map = courses.set_index('course_id')['course_name'].to_dict()

print(f"Student nodes: {data['student'].x.shape[0]}")

Student nodes: 2000


In [36]:
raw_means = {
    'gpa_start': students['gpa_start'].mean(),
    'effort_level': students['effort_level'].mean(),
    'risk_factor': students['risk_factor'].mean(),
    'avg_confusion': chatbot.groupby('student_id')['confusion_score'].mean().mean(),
    'avg_questions': chatbot.groupby('student_id')['num_questions'].mean().mean()
}

raw_stds = {
    'gpa_start': students['gpa_start'].std(),
    'effort_level': students['effort_level'].std(),
    'risk_factor': students['risk_factor'].std(),
    'avg_confusion': chatbot.groupby('student_id')['confusion_score'].mean().std(),
    'avg_questions': chatbot.groupby('student_id')['num_questions'].mean().std()
}

In [37]:
def add_cold_start_student(data, student_features, course_grades, course_id_map):
    sim = copy.deepcopy(data)
    sim['student'].x = torch.cat([sim['student'].x, student_features], dim = 0)
    new_idx = sim['student'].x.shape[0] - 1

    for course_id, grade in course_grades.items():
        c_idx = course_id_map[course_id]
        new_src = torch.tensor([new_idx], dtype = torch.long)
        new_dst = torch.tensor([c_idx], dtype = torch.long)
        new_weight = torch.tensor([1.0], dtype = torch.float)

        sim['student', 'enrolled_in', 'course'].edge_index = torch.cat([
            sim['student', 'enrolled_in', 'course'].edge_index,
            torch.stack([new_src, new_dst])
        ], dim = 1)
        sim['student', 'enrolled_in', 'course'].edge_weight = torch.cat([
            sim['student', 'enrolled_in', 'course'].edge_weight, new_weight
        ])
        sim['course', 'rev_enrolled_in', 'student'].edge_index = torch.cat([
            sim['course', 'rev_enrolled_in', 'student'].edge_index,
            torch.stack([new_dst, new_src])
        ], dim=1)
        sim['course', 'rev_enrolled_in', 'student'].edge_weight = torch.cat([
            sim['course', 'rev_enrolled_in', 'student'].edge_weight, new_weight
        ])

    return sim, new_idx

In [38]:
def get_knn_prior(new_gpa, course_id, n_neighbors = 50):
    students['gpa_diff'] = abs(students['gpa_start'] - new_gpa)
    similar = students.nsmallest(n_neighbors, 'gpa_diff')['student_id']

    similar_enrollments = enrollments[
        (enrollments['student_id'].isin(similar)) &
        (enrollments['course_id'] == course_id)
    ]

    if len(similar_enrollments) == 0:
        return torch.tensor([0.33, 0.34, 0.33])
    
    risk_dist = similar_enrollments['risk_class'].value_counts(normalize = True)
    return torch.tensor([
        risk_dist.get('Low', 0.0),
        risk_dist.get('Medium', 0.0),
        risk_dist.get('High', 0.0)
    ], dtype = torch.float)

In [39]:
def predict_blended(model, sim_data, student_idx, course_id, n_edges, new_gpa):
    knn_prior = get_knn_prior(new_gpa, course_id)

    with torch.no_grad():
        out = model(sim_data)
        s_emb = out[0][student_idx]
        c_emb = out[1][course_id_map[course_id]]
        pair = torch.cat([s_emb, c_emb, s_emb - c_emb, s_emb * c_emb]).unsqueeze(0)
        logits = model.task_a_head(pair)
        model_probs = torch.softmax(logits, dim = 1).squeeze()

    if n_edges == 0:
        alpha = 1.0
    elif n_edges <= 10:
        alpha = 0.6
    else:
        alpha = 0.1

    blended = alpha*knn_prior + (1-alpha) * model_probs
    pred = ['Low', 'Medium', 'High'][blended.argmax().item()]

    return pred, blended, knn_prior, model_probs

In [40]:
strong_features = torch.tensor([[
    (3.2 - raw_means['gpa_start']) / raw_stds['gpa_start'],
    (0.8 - raw_means['effort_level']) / raw_stds['effort_level'],
    (0.2 - raw_means['risk_factor']) / raw_stds['risk_factor'],
    (0.2 - raw_means['avg_confusion']) / raw_stds['avg_confusion'],
    (3.0 - raw_means['avg_questions']) / raw_stds['avg_questions']
]], dtype=torch.float)

med_features = torch.tensor([[
    (2.2 - raw_means['gpa_start']) / raw_stds['gpa_start'],
    (0.5 - raw_means['effort_level']) / raw_stds['effort_level'],
    (0.45 - raw_means['risk_factor']) / raw_stds['risk_factor'],
    (0.4 - raw_means['avg_confusion']) / raw_stds['avg_confusion'],
    (5.0 - raw_means['avg_questions']) / raw_stds['avg_questions']
]], dtype=torch.float)

weak_features = torch.tensor([[
    (1.5 - raw_means['gpa_start']) / raw_stds['gpa_start'],
    (0.3 - raw_means['effort_level']) / raw_stds['effort_level'],
    (0.7 - raw_means['risk_factor']) / raw_stds['risk_factor'],
    (0.6 - raw_means['avg_confusion']) / raw_stds['avg_confusion'],
    (8.0 - raw_means['avg_questions']) / raw_stds['avg_questions']
]], dtype=torch.float)

In [41]:
print("COLD START - 0 edges")

for name, features, gpa in [('Strong (GPA 3.2)', strong_features, 3.2),
                              ('Medium (GPA 2.2)', med_features, 2.2),
                              ('Weak (GPA 1.5)', weak_features, 1.5)]:
    sim = copy.deepcopy(data)
    sim['student'].x = torch.cat([sim['student'].x, features], dim = 0)
    idx = sim['student'].x.shape[0] - 1

    print(f"\n {name}:")
    for cid in ['C001', 'C005', 'C010', 'C023', 'C024', 'C033']:
        pred, blended, knn, model_p =predict_blended(model, sim, idx, cid, n_edges = 0, new_gpa = gpa)
        print(f"    {course_name_map[cid]:30s} → {pred:6s}  "
              f"B[L:{blended[0]:.2f} M:{blended[1]:.2f} H:{blended[2]:.2f}]")

COLD START - 0 edges

 Strong (GPA 3.2):
    Intro to Programming           → Low     B[L:1.00 M:0.00 H:0.00]
    Intro to Databases             → Low     B[L:1.00 M:0.00 H:0.00]
    Linear Algebra                 → Low     B[L:1.00 M:0.00 H:0.00]
    Machine Learning               → Low     B[L:0.82 M:0.18 H:0.00]
    Deep Learning                  → Low     B[L:0.74 M:0.26 H:0.00]
    AI Systems Capstone            → Low     B[L:0.75 M:0.25 H:0.00]

 Medium (GPA 2.2):
    Intro to Programming           → Medium  B[L:0.11 M:0.89 H:0.00]
    Intro to Databases             → Medium  B[L:0.08 M:0.92 H:0.00]
    Linear Algebra                 → Medium  B[L:0.21 M:0.79 H:0.00]
    Machine Learning               → Medium  B[L:0.00 M:0.95 H:0.05]
    Deep Learning                  → Medium  B[L:0.00 M:1.00 H:0.00]
    AI Systems Capstone            → Medium  B[L:0.00 M:1.00 H:0.00]

 Weak (GPA 1.5):
    Intro to Programming           → Medium  B[L:0.00 M:0.69 H:0.31]
    Intro to Databases  

In [42]:
print("=" * 70)
print("AFTER SEMESTER 1+2 — 10 edges (60% KNN, 40% model)")
print("=" * 70)

all_grades = {
    'Strong (GPA 3.2)': (strong_features, 3.2, {
        'C001': 3.5, 'C002': 3.2, 'C003': 3.4, 'C004': 3.7, 'C005': 3.8,
        'C006': 3.1, 'C007': 3.0, 'C008': 3.6, 'C009': 3.3, 'C010': 2.9,
    }),
    'Medium (GPA 2.2)': (med_features, 2.2, {
        'C001': 2.5, 'C002': 2.0, 'C003': 2.3, 'C004': 2.4, 'C005': 2.6,
        'C006': 1.9, 'C007': 2.1, 'C008': 2.7, 'C009': 2.2, 'C010': 1.8,
    }),
    'Weak (GPA 1.5)': (weak_features, 1.5, {
        'C001': 1.8, 'C002': 1.2, 'C003': 1.6, 'C004': 1.7, 'C005': 2.0,
        'C006': 1.3, 'C007': 1.4, 'C008': 2.1, 'C009': 1.5, 'C010': 1.1,
    }),
}

for name, (features, gpa, grades) in all_grades.items():
    sim, idx = add_cold_start_student(data, features, grades, course_id_map)
    print(f"\n  {name}:")
    for cid in ['C011', 'C013', 'C023', 'C024', 'C033']:
        pred, blended, knn, model_p = predict_blended(model, sim, idx, cid, n_edges=10, new_gpa=gpa)
        print(f"    {course_name_map[cid]:30s} → {pred:6s}  "
              f"B[L:{blended[0]:.2f} M:{blended[1]:.2f} H:{blended[2]:.2f}]  "
              f"KNN[L:{knn[0]:.2f} M:{knn[1]:.2f} H:{knn[2]:.2f}]  "
              f"Model[L:{model_p[0]:.2f} M:{model_p[1]:.2f} H:{model_p[2]:.2f}]")

AFTER SEMESTER 1+2 — 10 edges (60% KNN, 40% model)

  Strong (GPA 3.2):
    Data Structures                → Low     B[L:0.89 M:0.02 H:0.09]  KNN[L:1.00 M:0.00 H:0.00]  Model[L:0.73 M:0.04 H:0.23]
    Statistical Inference          → Low     B[L:0.89 M:0.02 H:0.09]  KNN[L:1.00 M:0.00 H:0.00]  Model[L:0.71 M:0.05 H:0.23]
    Machine Learning               → Low     B[L:0.76 M:0.14 H:0.10]  KNN[L:0.82 M:0.18 H:0.00]  Model[L:0.66 M:0.08 H:0.26]
    Deep Learning                  → Low     B[L:0.71 M:0.19 H:0.10]  KNN[L:0.74 M:0.26 H:0.00]  Model[L:0.68 M:0.08 H:0.25]
    AI Systems Capstone            → Low     B[L:0.69 M:0.20 H:0.11]  KNN[L:0.75 M:0.25 H:0.00]  Model[L:0.61 M:0.12 H:0.27]

  Medium (GPA 2.2):
    Data Structures                → Medium  B[L:0.10 M:0.76 H:0.14]  KNN[L:0.10 M:0.90 H:0.00]  Model[L:0.11 M:0.54 H:0.35]
    Statistical Inference          → Medium  B[L:0.09 M:0.76 H:0.15]  KNN[L:0.08 M:0.92 H:0.00]  Model[L:0.10 M:0.53 H:0.37]
    Machine Learning            

In [43]:
print("=" * 70)
print("AFTER SEMESTER 1+2+3 — 15 edges (10% KNN, 90% model)")
print("=" * 70)

all_grades_15 = {
    'Strong (GPA 3.2)': (strong_features, 3.2, {
        'C001': 3.5, 'C002': 3.2, 'C003': 3.4, 'C004': 3.7, 'C005': 3.8,
        'C006': 3.1, 'C007': 3.0, 'C008': 3.6, 'C009': 3.3, 'C010': 2.9,
        'C011': 3.0, 'C012': 2.8, 'C013': 2.9, 'C014': 3.2, 'C015': 3.1,
    }),
    'Medium (GPA 2.2)': (med_features, 2.2, {
        'C001': 2.5, 'C002': 2.0, 'C003': 2.3, 'C004': 2.4, 'C005': 2.6,
        'C006': 1.9, 'C007': 2.1, 'C008': 2.7, 'C009': 2.2, 'C010': 1.8,
        'C011': 2.1, 'C012': 1.8, 'C013': 2.0, 'C014': 2.3, 'C015': 2.2,
    }),
    'Weak (GPA 1.5)': (weak_features, 1.5, {
        'C001': 1.8, 'C002': 1.2, 'C003': 1.6, 'C004': 1.7, 'C005': 2.0,
        'C006': 1.3, 'C007': 1.4, 'C008': 2.1, 'C009': 1.5, 'C010': 1.1,
        'C011': 1.3, 'C012': 1.0, 'C013': 1.2, 'C014': 1.6, 'C015': 1.5,
    }),
}

for name, (features, gpa, grades) in all_grades_15.items():
    sim, idx = add_cold_start_student(data, features, grades, course_id_map)
    print(f"\n  {name}:")
    for cid in ['C023', 'C024', 'C028', 'C029', 'C033']:
        pred, blended, knn, model_p = predict_blended(model, sim, idx, cid, n_edges=15, new_gpa=gpa)
        print(f"    {course_name_map[cid]:30s} → {pred:6s}  "
              f"B[L:{blended[0]:.2f} M:{blended[1]:.2f} H:{blended[2]:.2f}]  "
              f"Model[L:{model_p[0]:.2f} M:{model_p[1]:.2f} H:{model_p[2]:.2f}]")

AFTER SEMESTER 1+2+3 — 15 edges (10% KNN, 90% model)

  Strong (GPA 3.2):
    Machine Learning               → Low     B[L:0.68 M:0.09 H:0.23]  Model[L:0.67 M:0.08 H:0.26]
    Deep Learning                  → Low     B[L:0.69 M:0.09 H:0.22]  Model[L:0.68 M:0.07 H:0.25]
    Computer Vision                → Low     B[L:0.72 M:0.06 H:0.22]  Model[L:0.68 M:0.07 H:0.25]
    NLP Fundamentals               → Low     B[L:0.70 M:0.08 H:0.23]  Model[L:0.66 M:0.08 H:0.25]
    AI Systems Capstone            → Low     B[L:0.64 M:0.12 H:0.24]  Model[L:0.62 M:0.11 H:0.27]

  Medium (GPA 2.2):
    Machine Learning               → Medium  B[L:0.08 M:0.51 H:0.41]  Model[L:0.09 M:0.46 H:0.45]
    Deep Learning                  → Medium  B[L:0.08 M:0.53 H:0.39]  Model[L:0.09 M:0.48 H:0.43]
    Computer Vision                → Medium  B[L:0.08 M:0.58 H:0.35]  Model[L:0.09 M:0.53 H:0.39]
    NLP Fundamentals               → Medium  B[L:0.09 M:0.53 H:0.38]  Model[L:0.09 M:0.48 H:0.43]
    AI Systems Capstone